# 1 — Setup
One-time qualification for *Image Enhancement as Preprocessing for Industrial Anomaly Detection Under Challenging Illumination*.
Run **Runtime → Run all** in a fresh Colab CPU runtime and authorize Drive access. Only the output directory is written.
The summary is marked incomplete before inspection so an interrupted run cannot leave a current PASS.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv
import json
import os
import platform
import PIL
from PIL import Image

# All configurable paths are here; do not point output inside a dataset.
dataRoot = Path('/content/drive/MyDrive/ColabData/CS7367_Vision')
datasetRoots = {'MVTec AD 2': dataRoot / 'mvtec_ad_2',
                'VisA': dataRoot / 'VisA_20220922'}
outputRoot = dataRoot / 'output' / 'validation'
for datasetRoot in datasetRoots.values():
    if outputRoot.resolve().is_relative_to(datasetRoot.resolve()):
        raise ValueError('Output must be outside both dataset roots.')
outputRoot.mkdir(parents=True, exist_ok=True)
runStartedUtc = datetime.now(timezone.utc).isoformat()
exceptions = []
inventory = []
imageRecords = []
maskRecords = []
maskPairs = []
annotationCounts = Counter()
filesChecked = {name: Counter() for name in datasetRoots}
inventoryComplete = {name: False for name in datasetRoots}
integrityComplete = {name: False for name in datasetRoots}
accessibleRoots = {}
limitations = {
    'MVTec AD 2': ['Private test ground truth is intentionally non-public. Missing private image partitions are recorded as scope limitations; present private partitions must be nonempty and readable.'],
    'VisA': ['Original release has Normal/Anomaly groups, not native train/test partitions. Normal-image masks are intentionally omitted. No split is created.']}
Image.init()
imageExtensions = set(Image.registered_extensions())

def addIssue(dataset, category, partition, filePath, description, severity='critical'):
    exceptions.append(dict(dataset=dataset, category=category, partition=partition,
                           filePath=str(filePath), severity=severity, issueDescription=description))

def listImages(folder):
    foundFiles = []
    def raiseWalkError(error):
        raise error
    if folder.is_dir():
        for currentRoot, folderNames, fileNames in os.walk(folder, onerror=raiseWalkError):
            folderNames.sort()
            for fileName in sorted(fileNames):
                filePath = Path(currentRoot) / fileName
                if not fileName.startswith('.'):
                    # Include unknown extensions too: an unexpected file cannot silently escape decoding.
                    foundFiles.append(filePath)
    return foundFiles

def findRoot(root, expectedCategories):
    # Permit an extraction wrapper, but never select between ambiguous copies.
    candidates = [root] if root.is_dir() and any((root / c).is_dir() for c in expectedCategories) else []
    if root.is_dir() and not candidates:
        candidates = sorted({p.parent for c in expectedCategories for p in root.rglob(c) if p.is_dir()})
    if len(candidates) != 1:
        raise ValueError(f'Expected one category root below {root}; found {len(candidates)}.')
    return candidates[0]

def addGroup(dataset, category, partition, folder, required=True):
    groupImages = listImages(folder)
    if required and not groupImages:
        addIssue(dataset, category, partition, folder, 'Required image directory is missing or empty.')
    imageRecords.extend((dataset, category, partition, p) for p in groupImages)
    inventory.append(dict(dataset=dataset, category=category, partition=partition,
                          imageCount=len(groupImages), annotationCount=0, maskCount=0))
    return groupImages

# Invalidate earlier evidence before any inventory work.
for csvName, fieldNames in [('dataset_inventory.csv', ['dataset', 'category', 'partition', 'imageCount', 'annotationCount', 'maskCount']),
                            ('validation_exceptions.csv', ['dataset', 'category', 'partition', 'filePath', 'severity', 'issueDescription'])]:
    with (outputRoot / csvName).open('w', newline='', encoding='utf-8') as handle:
        csv.DictWriter(handle, fieldnames=fieldNames).writeheader()
with (outputRoot / 'validation_summary.json').open('w', encoding='utf-8') as handle:
    json.dump({'validationTimestampUtc': runStartedUtc, 'complete': False,
               'overallResult': 'FAIL', 'reason': 'Validation started; completion evidence not yet written.'}, handle, indent=2)
for dataset, root in datasetRoots.items():
    try:
        if not root.is_dir():
            raise FileNotFoundError(root)
        next(root.iterdir(), None)
        print(f'{dataset}: accessible — {root}')
    except Exception as error:
        addIssue(dataset, '', '', root, f'Dataset root inaccessible: {error}')
        print(f'{dataset}: FAIL — root inaccessible')


# 2 — Dataset Inventory
[MVTec documentation](https://www.mvtec.com/research-teaching/datasets/mvtec-ad-2) and its linked official code utilities define `train/good`, `validation/good`, `test_public/{good,bad}`, and flat private image partitions. Public anomaly masks are `test_public/ground_truth/bad/<stem>_mask.png`.
Primary categories: **can, wallplugs, sheet_metal**. All eight categories are checked.
[VisA documentation](https://github.com/amazon-science/spot-diff) defines the original category-level `Data/Images/{Normal,Anomaly}`, `Data/Masks/Anomaly`, and `image_anno.csv` (also called `image_annot.csv` in its prose). All 12 official categories and any additional discovered categories are inventoried. Private MVTec image partitions may be absent in a public-only installation; this limits qualification scope and is explicitly reported.


In [ ]:
mvtecCategories = ['can', 'fabric', 'fruit_jelly', 'rice', 'sheet_metal', 'vial', 'wallplugs', 'walnuts']
visaCategories = ['candle', 'capsules', 'cashew', 'chewinggum', 'fryum', 'macaroni1', 'macaroni2', 'pcb1', 'pcb2', 'pcb3', 'pcb4', 'pipe_fryum']
visaCategoryRoots = []
for dataset, expectedCategories in [('MVTec AD 2', mvtecCategories), ('VisA', visaCategories)]:
    try:
        root = findRoot(datasetRoots[dataset], expectedCategories)
        accessibleRoots[dataset] = root
        if dataset == 'MVTec AD 2':
            discoveredCategories = {p.name for p in root.iterdir() if p.is_dir() and any((p / s).is_dir() for s in ['train', 'validation', 'test_public', 'test_private', 'test_private_mixed'])}
        else:
            discoveredCategories = {p.name for p in root.iterdir() if p.is_dir() and (p / 'Data').is_dir()}
        for category in sorted(set(expectedCategories) | discoveredCategories):
            categoryRoot = root / category
            if not categoryRoot.is_dir():
                addIssue(dataset, category, '', categoryRoot, 'Missing required category.')
            if category not in expectedCategories:
                addIssue(dataset, category, '', categoryRoot, 'Unexpected category; requires review.')
            if dataset == 'MVTec AD 2':
                for partition in ['train', 'validation', 'test_public', 'test_private', 'test_private_mixed']:
                    partitionRoot = categoryRoot / partition
                    if 'private' in partition:
                        if not partitionRoot.exists():
                            addIssue(dataset, category, partition, partitionRoot, 'Private image partition unavailable; excluded from qualification scope.', 'limitation')
                            inventory.append(dict(dataset=dataset, category=category, partition=partition, imageCount=0, annotationCount=0, maskCount=0))
                            continue
                        groupImages = addGroup(dataset, category, partition, partitionRoot)
                        for imagePath in groupImages:
                            if imagePath.parent != partitionRoot:
                                addIssue(dataset, category, partition, imagePath, 'Private images must be directly inside the partition.')
                    else:
                        groupImages = []
                        for label in (['good', 'bad'] if partition == 'test_public' else ['good']):
                            groupImages += addGroup(dataset, category, partition + '/' + label, partitionRoot / label)
                        maskRoot = partitionRoot / 'ground_truth'
                        for maskPath in listImages(maskRoot):
                            maskRecords.append((dataset, category, partition + '/bad', maskPath))
                        for imagePath in groupImages:
                            if imagePath.parent.name == 'bad':
                                maskPath = maskRoot / 'bad' / (imagePath.stem + '_mask.png')
                                maskPairs.append((dataset, category, partition + '/bad', imagePath, maskPath))
                        expectedImages = set(groupImages)
                        for imagePath in listImages(partitionRoot):
                            if not imagePath.is_relative_to(maskRoot) and imagePath not in expectedImages:
                                imageRecords.append((dataset, category, partition, imagePath))
                                inventory.append(dict(dataset=dataset, category=category, partition=partition, imageCount=1, annotationCount=0, maskCount=0))
                                addIssue(dataset, category, partition, imagePath, 'Image outside the documented good/bad layout.')
            else:
                visaCategoryRoots.append(categoryRoot)
                for label in ['Normal', 'Anomaly']:
                    addGroup(dataset, category, label, categoryRoot / 'Data' / 'Images' / label)
                for maskPath in listImages(categoryRoot / 'Data' / 'Masks'):
                    maskRecords.append((dataset, category, 'Anomaly', maskPath))
                knownImages = {r[3] for r in imageRecords if r[0] == dataset and r[1] == category}
                for imagePath in listImages(categoryRoot / 'Data' / 'Images'):
                    if imagePath not in knownImages:
                        imageRecords.append((dataset, category, 'unexpected', imagePath))
                        inventory.append(dict(dataset=dataset, category=category, partition='unexpected', imageCount=1, annotationCount=0, maskCount=0))
                        addIssue(dataset, category, '', imagePath, 'Image outside Normal/Anomaly groups.')
        inventoryComplete[dataset] = True
    except Exception as error:
        addIssue(dataset, '', '', datasetRoots[dataset], f'Inventory incomplete: {type(error).__name__}: {error}')
for dataset in datasetRoots:
    print(f'{dataset}: {sum(r["imageCount"] for r in inventory if r["dataset"] == dataset)} images inventoried')


# 3 — Dataset Integrity
Decode each image and available mask individually with Pillow. Validate CSV image/label/mask references, annotation coverage, duplicate references, mask pairing, and dimensions. VisA masks may contain multiple anomaly IDs; they are not required to be binary. Missing masks for normal images or withheld private labels are not failures.


In [ ]:
def resolveReference(value, categoryRoot):
    value = value.strip().replace('\\', '/')
    relativePath = Path(value)
    if not value or relativePath.is_absolute() or '..' in relativePath.parts:
        raise ValueError(f'Invalid relative reference: {value!r}')
    candidates = {(base / relativePath).resolve() for base in [categoryRoot, categoryRoot.parent]
                  if (base / relativePath).is_file()}
    candidates = {p for p in candidates if p.is_relative_to(categoryRoot.resolve())}
    if len(candidates) != 1:
        raise ValueError(f'Reference missing, outside category, or ambiguous: {value}')
    return candidates.pop()

for categoryRoot in visaCategoryRoots:
    category = categoryRoot.name
    try:
        csvPaths = [p for p in [categoryRoot / 'image_anno.csv', categoryRoot / 'image_annot.csv'] if p.is_file()]
        if len(csvPaths) != 1:
            raise ValueError('Expected exactly one per-category annotation CSV.')
        categoryImages = {p.resolve(): partition for dataset, cat, partition, p in imageRecords if dataset == 'VisA' and cat == category}
        seenImages = set()
        with csvPaths[0].open(newline='', encoding='utf-8-sig') as handle:
            reader = csv.DictReader(handle)
            headerMap = {str(h).strip().lower(): h for h in (reader.fieldnames or [])}
            imageKey = next((headerMap[k] for k in ['image', 'image_path'] if k in headerMap), None)
            maskKey = next((headerMap[k] for k in ['mask', 'mask_path'] if k in headerMap), None)
            labelKey = headerMap.get('label')
            if not all([imageKey, maskKey, labelKey]):
                raise ValueError(f'Unrecognized annotation columns: {reader.fieldnames}')
            for rowNumber, row in enumerate(reader, 2):
                try:
                    imagePath = resolveReference(row[imageKey], categoryRoot)
                    label = row[labelKey].strip().lower()
                    partition = {'normal': 'Normal', 'anomaly': 'Anomaly'}.get(label)
                    if partition is None or categoryImages.get(imagePath) != partition:
                        raise ValueError('Invalid label or image reference disagrees with image group.')
                    if imagePath in seenImages:
                        raise ValueError('Duplicate image annotation.')
                    seenImages.add(imagePath)
                    annotationCounts[('VisA', category, partition)] += 1
                    maskValue = (row[maskKey] or '').strip()
                    if label == 'normal' and maskValue.lower() in ['', 'none', 'nan']:
                        maskValue = ''
                    if label == 'anomaly' or maskValue:
                        maskPath = resolveReference(maskValue, categoryRoot)
                        if not maskPath.is_relative_to((categoryRoot / 'Data' / 'Masks').resolve()):
                            raise ValueError('Mask reference is outside Data/Masks.')
                        maskPairs.append(('VisA', category, partition, imagePath, maskPath))
                except Exception as error:
                    addIssue('VisA', category, '', csvPaths[0], f'CSV row {rowNumber}: {error}')
        for imagePath in categoryImages.keys() - seenImages:
            addIssue('VisA', category, categoryImages[imagePath], imagePath, 'Image has no valid CSV annotation.')
    except Exception as error:
        addIssue('VisA', category, '', categoryRoot, f'Annotation validation failed: {error}')

imageSizes = {}
maskSizes = {}
for recordType, records, sizeMap in [('images', imageRecords, imageSizes), ('masks', maskRecords, maskSizes)]:
    seenFiles = set()
    for dataset, category, partition, filePath in records:
        fileKey = (dataset, filePath.resolve())
        if fileKey in seenFiles:
            continue
        seenFiles.add(fileKey)
        filesChecked[dataset][recordType] += 1
        try:
            with Image.open(filePath) as image:
                image.verify()
            with Image.open(filePath) as image:
                image.load()
                sizeMap[filePath.resolve()] = image.size
        except Exception as error:
            addIssue(dataset, category, partition, filePath, f'Unreadable {recordType}: {type(error).__name__}: {error}')

pairedMasks = Counter()
for dataset, category, partition, imagePath, maskPath in maskPairs:
    imagePath, maskPath = imagePath.resolve(), maskPath.resolve()
    pairedMasks[(dataset, maskPath)] += 1
    if not maskPath.is_file():
        addIssue(dataset, category, partition, maskPath, 'Required anomaly mask is missing.')
    elif maskPath not in maskSizes:
        addIssue(dataset, category, partition, maskPath, 'Referenced mask is unreadable or outside mask inventory.')
    elif imagePath in imageSizes and imageSizes[imagePath] != maskSizes[maskPath]:
        addIssue(dataset, category, partition, maskPath, f'Image/mask dimensions disagree: {imageSizes[imagePath]} versus {maskSizes[maskPath]}.')
for dataset, category, partition, maskPath in maskRecords:
    count = pairedMasks[(dataset, maskPath.resolve())]
    if count != 1:
        addIssue(dataset, category, partition, maskPath, f'Mask must match exactly one image; found {count}.')
for row in inventory:
    key = (row['dataset'], row['category'], row['partition'])
    row['annotationCount'] = annotationCounts[key]
    if row['dataset'] == 'MVTec AD 2' and 'private' not in row['partition']:
        row['annotationCount'] = row['imageCount']  # labels encoded by good/bad directory
    row['maskCount'] = sum(1 for record in maskRecords if record[:3] == key)
for dataset in datasetRoots:
    integrityComplete[dataset] = inventoryComplete[dataset]
    print(f'{dataset}: checked {filesChecked[dataset]["images"]} images and {filesChecked[dataset]["masks"]} masks')


# 4 — Acceptance and Evidence
PASS requires completed inventory and integrity checks with zero critical failures. Private image exclusions are stated in the summary and do not qualify unavailable data. No dataset version is inferred from a folder name alone.
The three files below are machine-generated validation evidence. Carlos runs the notebook; Sam Chatley reviews the artifacts and recommends acceptance; Carlos retains final acceptance authority.


In [ ]:
validationSummary = {
    'validationTimestampUtc': datetime.now(timezone.utc).isoformat(),
    'runStartedUtc': runStartedUtc,
    'complete': all(inventoryComplete.values()) and all(integrityComplete.values()),
    'runtime': {'python': platform.python_version(), 'pillow': PIL.__version__},
    'datasets': {}}
for dataset, configuredRoot in datasetRoots.items():
    datasetRows = [r for r in inventory if r['dataset'] == dataset]
    criticalCount = sum(e['dataset'] == dataset and e['severity'] == 'critical' for e in exceptions)
    complete = inventoryComplete[dataset] and integrityComplete[dataset]
    datasetLimitations = limitations[dataset] + [e['issueDescription'] + ' ' + e['filePath'] for e in exceptions if e['dataset'] == dataset and e['severity'] == 'limitation']
    validationSummary['datasets'][dataset] = {
        'configuredRoot': str(configuredRoot),
        'accessibleRoot': str(accessibleRoots[dataset]) if dataset in accessibleRoots else None,
        'releaseIdentifier': None,
        'releaseNote': ('MVTec AD 2; installed release not independently verified.' if dataset == 'MVTec AD 2' else 'Official AWS archive is VisA_20220922.tar; installed release not independently verified.'),
        'complete': complete,
        'result': 'PASS' if complete and criticalCount == 0 else 'FAIL',
        'filesChecked': dict(filesChecked[dataset]),
        'categoryCount': len({r['category'] for r in datasetRows if r['imageCount']}),
        'partitionCount': len({r['partition'] for r in datasetRows if r['imageCount']}),
        'categoryPartitionCount': sum(r['imageCount'] > 0 for r in datasetRows),
        'criticalFailureCount': criticalCount,
        'annotationAvailabilityOrLimitations': datasetLimitations,
        'primaryResearchCategories': ['can', 'wallplugs', 'sheet_metal'] if dataset == 'MVTec AD 2' else []}
validationSummary['overallResult'] = 'PASS' if all(d['result'] == 'PASS' for d in validationSummary['datasets'].values()) else 'FAIL'
for csvName, rows, fieldNames in [
    ('dataset_inventory.csv', inventory, ['dataset', 'category', 'partition', 'imageCount', 'annotationCount', 'maskCount']),
    ('validation_exceptions.csv', exceptions, ['dataset', 'category', 'partition', 'filePath', 'severity', 'issueDescription'])]:
    with (outputRoot / csvName).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldNames)
        writer.writeheader()
        writer.writerows(rows)
# Replace checkpoint only after both CSVs have been saved.
summaryTemp = outputRoot / 'validation_summary.json.tmp'
with summaryTemp.open('w', encoding='utf-8') as handle:
    json.dump(validationSummary, handle, indent=2)
summaryTemp.replace(outputRoot / 'validation_summary.json')
for dataset, result in validationSummary['datasets'].items():
    print(f"{dataset}: {result['result']} | {result['filesChecked'].get('images', 0)} images | {result['criticalFailureCount']} critical failures")
print('Overall:', validationSummary['overallResult'])
print('Evidence:', outputRoot)
